# <span style="color:#2563EB">Building the First Classical Baseline</span>

<table width="100%" cellpadding="10" cellspacing="0">
<tr bgcolor="#f8fafc"><td>
<p><b>Fundamentals of Natural Language Processing</b> | Universitat Autonoma de Barcelona | 2025 2026</p>
<p>Phoebe Iglesias (1713459), David Redrejo (1790336), Pau Rossell (1750424)</p>
<h3><font color="#2563EB">Notebook question</font></h3>
<p>How far can a clean classical pipeline go before we tune anything?</p>
<h3><font color="#2563EB">Connection with the previous notebook</font></h3>
<p>This follows <b><font color="#0F766E">Notebook 01: Reading the Ground Before Modeling</font></b>. The EDA showed that short text, normalization, and character fragments are the main signal.</p>
<h3><font color="#2563EB">What this chapter contributes</font></h3>
<p>We turn the EDA into a reproducible baseline and generate the first submission file.</p>
</td></tr>
</table>

<table width="100%" cellpadding="8" cellspacing="0">
<tr bgcolor="#2563EB"><th><font color="white">Move</font></th><th><font color="white">What we try to understand</font></th></tr>
<tr><td>1</td><td>Setup</td></tr><tr><td>2</td><td>Load data</td></tr><tr><td>3</td><td>Normalize</td></tr><tr><td>4</td><td>Build target</td></tr><tr><td>5</td><td>Split</td></tr><tr><td>6</td><td>TF IDF</td></tr><tr><td>7</td><td>SVM</td></tr><tr><td>8</td><td>Validate</td></tr><tr><td>9</td><td>Submit</td></tr>
</table>

## Chapter Map

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

chapter_color = "#2563EB"
labels = ['Setup', 'Load data', 'Normalize', 'Build target', 'Split', 'TF IDF', 'SVM', 'Validate', 'Submit']

fig, ax = plt.subplots(figsize=(14, 2.6))
ax.set_xlim(0, len(labels))
ax.set_ylim(0, 1)
ax.axis("off")

for idx, label in enumerate(labels):
    card = FancyBboxPatch(
        (idx + 0.06, 0.25), 0.88, 0.48,
        boxstyle="round,pad=0.04,rounding_size=0.05",
        linewidth=1.4,
        edgecolor=chapter_color,
        facecolor="#f8fafc"
    )
    ax.add_patch(card)
    ax.text(idx + 0.5, 0.49, label, ha="center", va="center", fontsize=10.5, color="#0f172a", wrap=True)
    if idx < len(labels) - 1:
        ax.annotate("", xy=(idx + 1.02, 0.49), xytext=(idx + 0.94, 0.49), arrowprops=dict(arrowstyle=">", color=chapter_color, lw=1.8))

ax.text(0.02, 0.9, "How this notebook moves", fontsize=14, weight="bold", color=chapter_color)
plt.show()

## Setup and Execution Context

We first make the notebook robust to where it is opened from. This is small but important: if paths fail, the model story stops before it begins. This baseline is CPU based because sparse TF IDF and LinearSVC do not use the GPU.

In [ ]:
import os
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_processing import (
    normalize_text,
    normalize_texts,
    extract_category,
    prepare_category_dataset,
    split_category_dataset,
)
from evaluation import (
    calculate_category_metrics,
    print_classification_report,
    generate_submission,
)

required_files = [
    DATA_DIR / "codification_data.csv",
    DATA_DIR / "leaderboard_data.csv",
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required data files. Place the project CSV files in the data/ folder: "
        + ", ".join(missing)
    )

print(f"Project root : {PROJECT_ROOT}")
print(f"Data folder  : {DATA_DIR}")
print(f"Submissions  : {SUBMISSION_DIR}")
print("Setup complete.")

## Data Loading

With the environment ready, we load the supervised file and the leaderboard file. We look at counts and first rows because our first responsibility is to make sure the model is seeing the files we think it is seeing.

In [ ]:
codif_df = pd.read_csv(DATA_DIR / "codification_data.csv")
lead_df = pd.read_csv(DATA_DIR / "leaderboard_data.csv")

print(f"Training rows  : {len(codif_df):,}")
print(f"Unique codes   : {codif_df['Code'].nunique():,}")
print(f"Unique literals: {codif_df['Literal'].nunique():,}")
print(f"Leaderboard    : {len(lead_df):,}")
print()
display(codif_df.head())
display(lead_df.head())

assert {"Code", "Literal"}.issubset(codif_df.columns)
assert {"id", "Literal"}.issubset(lead_df.columns)
assert len(lead_df) > 0

## Literal Normalization

The EDA showed that formatting changes matter, so we normalize before vectorizing. We lowercase, strip accents, remove punctuation, keep digits, and collapse spaces. The sample strings let us check the transformation with our eyes before trusting it.

In [ ]:
samples = [
    "Bronchial hyperreactivity",
    "Dilated cardiomyopathy",
    "Atypical febrile seizure",
    "Breast prosthesis",
    "IBUPROFEN ALLERGY",
    "<font>Hepatitis C virus</font>",
    "Ventral hernia",
]

print(f"{'Original':<35} Normalized")
print("-" * 70)
for sample in samples:
    print(f"{sample:<35} {normalize_text(sample)}")


## Target Construction

Now we translate the full ICD code into the target used by the competition. We keep the first character, then resolve repeated literals by majority vote. This gives us one trainable label per literal while keeping the task aligned with the submission format.

In [ ]:
print("Code -> category examples:")
for code_value in ["J9809", "07CP0ZZ", "N801", "Z6740", "O99284", "BW03ZZZ"]:
    print(f"  {code_value:<10} -> {extract_category(code_value)}")

cat_df = prepare_category_dataset(
    codif_df,
    literal_col="Literal",
    code_col="Code",
)
cat_df["Literal"] = normalize_texts(cat_df["Literal"])

print()
print("Prepared category dataset:")
display(cat_df.head(10))
assert cat_df["Literal"].notna().all()
assert cat_df["y_category"].notna().all()

## Train Validation Split

Before training, we create a stratified split. We do this because category imbalance is real, and a careless split could make validation either too easy or too strange.

In [ ]:
X_train, X_val, y_train, y_val = split_category_dataset(
    cat_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print(f"Train labels: {len(set(y_train))}")
print(f"Val labels  : {len(set(y_val))}")

## Character TF IDF Features

We now convert literals into features. Character grams are a good match for short medical text because they capture roots, abbreviations, and spelling variants better than word tokens alone.

In [ ]:
tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 6),
    sublinear_tf=True,
    max_features=100_000,
    min_df=2,
    dtype=np.float32,
)

print("Fitting TF-IDF on training literals...")
t0 = time.time()
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
print(f"Done in {time.time() - t0:.1f}s")
print(f"Train matrix    : {X_train_tfidf.shape}")
print(f"Validation matrix: {X_val_tfidf.shape}")
print(f"Vocabulary size : {len(tfidf.vocabulary_):,}")

assert X_train_tfidf.shape[0] == len(X_train)
assert X_val_tfidf.shape[0] == len(X_val)

## Linear SVM Classifier

The first classifier is intentionally simple and strong. A linear SVM works well with sparse high dimensional features, and it gives us a baseline that is fast enough to inspect and repeat.

In [ ]:
print("Training LinearSVC baseline...")
t0 = time.time()

svm_clf = LinearSVC(
    C=1.0,
    max_iter=10_000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
svm_clf.fit(X_train_tfidf, y_train)

print(f"Done in {time.time() - t0:.1f}s")
print(f"Classes learned: {len(svm_clf.classes_)}")

## Validation Metrics

Once the model is trained, we read the validation metrics. Accuracy is the main target, but weighted F1 and macro F1 tell us whether the model is only doing well on frequent classes.

In [ ]:
y_pred_val = svm_clf.predict(X_val_tfidf)
metrics = calculate_category_metrics(y_val, y_pred_val)

print("SVM validation results")
print("-" * 30)
for metric_name, value in metrics.items():
    print(f"{metric_name:<20}: {value:.4f}")

assert 0.0 <= metrics["accuracy"] <= 1.0
assert len(y_pred_val) == len(y_val)

In [ ]:
print("Per-class classification report:")
print_classification_report(y_val, y_pred_val)

## Leaderboard Submission

Only after validation do we touch the leaderboard. We transform it with the already fitted vectorizer, predict categories, write the CSV, and check the shape and empty values.

In [ ]:
lead_literals_norm = normalize_texts(lead_df["Literal"].tolist())
X_lead_tfidf = tfidf.transform(lead_literals_norm)
y_lead_pred = svm_clf.predict(X_lead_tfidf)

print(f"Leaderboard TF-IDF matrix: {X_lead_tfidf.shape}")
print(f"Predictions             : {len(y_lead_pred):,}")
print(f"Unique categories       : {len(set(y_lead_pred))}")
print(f"Empty predictions       : {sum(1 for pred in y_lead_pred if not pred)}")

assert X_lead_tfidf.shape[0] == len(lead_df)
assert len(y_lead_pred) == len(lead_df)

In [ ]:
submission_path = SUBMISSION_DIR / "svm_baseline.csv"
submission_df = generate_submission(
    lead_df,
    y_lead_pred,
    output_path=str(submission_path),
)

expected_columns = ["id", "Literal", "y_category"]
assert list(submission_df.columns) == expected_columns
assert len(submission_df) == len(lead_df)
assert submission_df["y_category"].notna().all()
assert (submission_df["y_category"].astype(str).str.len() > 0).all()
assert submission_path.exists()

print()
print("Submission preview:")
display(submission_df.head(10))
print()
print("Category distribution in submission:")
print(submission_df["y_category"].value_counts().sort_index())

## Baseline Reading

This notebook gives us a working reference point. It also raises questions that become <b><font color="#16A34A">Notebook 03: Tuning the Baseline Until It Generalizes</font></b>: should we keep balanced weights, should we keep rare fragments, and are the n gram lengths right for such short text?